# **Data Visualisation (ST1502) CA1: Data Cleaning (Additional Data)**
Name: Goh Kun Ming

Class: DAAA/FT/2B/02

Admin Number: P2415691

Lecturer: Senior Lecturer Peter Wai Tong Lee

---
# 1. Environment Setup and Libraries Import

In this section, we will be importing the various relevant libraries required to clean the provided Data. We will also need to mount the Google Drive so as to allow for quick and efficient access to the provided dataset.

In [ ]:
"""DAVI CA1 — Basic Colab Setup & Utilities.

This module sets up display options, mounts Google Drive, and provides
lightweight styling helpers for DataFrame rendering in notebooks.
It also initialises project paths for raw and cleaned datasets.

Usage:
    • Run this cell once at the start of your session.
    • Use `RAW_CSV` and `CLEAN_CSV` as your canonical file paths.

Author: Goh Kun Ming
"""

# =========================
# Import Libraries
# =========================
from pathlib import Path
from typing import Optional, Any

import plotly.express as px
import plotly.io as pio
import pandas as pd
import numpy as np
import re
import os
import pandas as pd
import numpy as np
import matplotlib as mpl
try:
    from IPython.display import display
except ImportError:
    def display(obj: Any) -> None:
        print(obj)


# =========================
# Project Paths
# =========================
BASE_DIR: Path = Path(__file__).resolve().parents[1] if "__file__" in globals() else Path("./")
BASE_DIR.mkdir(parents=True, exist_ok=True)

RAW_CSV: Path = BASE_DIR / "data/raw/location_enriched_hdb_resale_raw.csv"
CLEAN_CSV: Path = BASE_DIR / "data/processed/location_enriched_hdb_resale_clean.csv"

With reference to the Code Cell above, we are able to confirm that the libraries and the Google Drive have been successfully imported and mounted and we are able to proceed to conduct the Data Cleaning process on the provided dataset.

---
# 2. Data Loading & Column Normalisation

In this section, we will be loading the provided HDB dataset into the environemnt in a dataframe. Subsequently, we will be normalising all the columns to a standardised format so as to ensure consistency throughout the features within the data. Finally, we will be previewing the dataframe to get a better sensing and understanding of what the dataset holds with reference to the contents each of the data column holds.

In [ ]:
"""DAVI CA1 — Data Loading & Column Normalisation.

Loads the raw HDB Resale Flat Prices dataset, standardises all column names
to snake_case, and displays a styled preview (top 10 rows) with a dark header
and blue gradient for dark-mode readability.
"""

# =========================
# Load Raw Dataset
# =========================
df = pd.read_csv(RAW_CSV, index_col=False)
df = df.loc[:, ~df.columns.str.contains('^Unnamed')]

# =========================
# Standardise Column Names
# =========================
def normalise_colname(s: str) -> str:
    """Convert a raw column name to lowercase snake_case.

    Removes whitespace, replaces spaces and hyphens with underscores,
    and lowercases all characters.

    Args:
        s: Raw column name.

    Returns:
        Standardised column name suitable for analysis.
    """
    return s.strip().lower().replace(" ", "_").replace("-", "_")


df.columns = [normalise_colname(c) for c in df.columns]


# =========================
# Display Preview
# =========================
def style_blue(df_in):
    """Return a styled DataFrame preview with a dark header.

    Applies a dark header background and a blue gradient on cells
    for better visibility in dark-themed environments.

    Args:
        df_in: DataFrame to style.

    Returns:
        A pandas Styler object for notebook display.
    """
    return (
        df_in.style.background_gradient(cmap="Blues", axis=None)
        .set_properties(**{
            "border": "1px solid #444",
            "font-size": "12px",
            "color": "#e8e8e8",
        })
        .set_table_styles([
            {
                "selector": "th",
                "props": [
                    ("background-color", "#1f2c3b"),
                    ("color", "#f1f1f1"),
                    ("border", "1px solid #555"),
                    ("font-weight", "bold")
                ]
            },
            {
                "selector": "td",
                "props": [
                    ("background-color", "#101820"),
                ]
            }
        ])
    )


# =========================
# Display Dataframe
# =========================
display(style_blue(df.head(10)))

With reference to the Code Cell above, we are able to verify that the dataset have been successfully imported into the environment. We are also able to make a few observations within the dataset with reference to the metadata provided. The observations are indicated below

**month**
- Metadata Reference:
> Given in the format of year-month. You may retrieve the year data from this column, which may be useful when analysing the time trend for HDB resale price.

- No inconsistencies observed.

- A potential feature engineering opportunitiy is observed where we are able to split the year and month so as to potentially conduct Time Series Analysis / Forecast.

**town**
- Metadata Reference:
> Town location should be one of the key factors affecting HDB resale price — you are generally expecting an HDB flat in Orchard has a much higher resale price than Yishun given the same flat type.

- Inconsistencies observed where there are both 'AMK' and 'ANG MO KIO'. There are most likely more inconsistencies like these which we will need to identify and address during the data cleaning stage.

- A data format standardisation approach may be required to upkeep the quality of the column where we change the current data entry format of full capitalisation to a data entry format of first-letter capitalisation (i.e., 'ANG MO KIO' to 'Ang Mo Kio').

**flat_type**
- Metadata Reference:
> There are 7 different kinds of flat types: 1 Room, 2 Room, 3 Room, 4 Room, 5 Room, EC and Multi-generation. Among which the 4 Room HDB flats are the most popular ones in Singapore.

- Inconsistencies observed where there are both '3-ROOM' and '3 ROOM'. There are most likely more inconsistencies like these which we will need to identify and address during the data cleaning stage.

- A data format standardisation approach may be required to upkeep the quality of the column where we change the current data entry format of full capitalisation to a data entry format of first-letter capitalisation (i.e., '3 ROOM' to '3 Room').

**block**
- Metadata Reference:
> The block number of the flat.

- Inconsistencies observed where there are block numbers with special characters such as '@@230' and '@406'. This is most likely due to data entry errors which we will need to indentify and address during the data cleaning stage.

**street_name**
- Metadata Reference:
> The street name where the flat is located.

- Inconsistencies observed where there are both 'AMK' and 'ANG MO KIO'. There are most likely more inconsistencies like these which we will need to identify and address during the data cleaning stage.

- A data format standardisation approach may be required to upkeep the quality of the column where we change the current data entry format of full capitalisation to a data entry format of first-letter capitalisation (i.e., 'ANG MO KIO' to 'Ang Mo Kio').

- During the Data Cleaning segment, we should also approach the issues of short-form used despite it being a common and standard way of writing address in Singapore (i.e., 'AVE' to 'Avenue').

**storey_range**
- Metadata Reference:
> The range of storeys where the flat is located. This column is given as a string rather than numbers.

- Inconsistencies observed where there are block numbers with special characters such as '01@03' and '10@12'. This is most likely due to data entry errors which we will need to indentify and address during the data cleaning stage.

- A data format standardisation approach may be required to upkeep the quality of the column where we change the current data entry format of full capitalisation to a data entry format of no capitalisation (i.e., 'TO' to 'to').

- A potential feature engineering opportunitiy is observed where we are able to split the starting storey and the last storey to conduct independent analysis.

**floor_area_sqm**
- Metadata Reference:
> The floor area of the flat in square meters.

- No inconsistencies observed.

**flat_model**
- Metadata Reference:
> There are different flat models. This factor would play a key role in the overall flat price. E.g., the DBSS (Design, Build and Sell Scheme) flats would have a higher resale price considering it allows buyers to design the HDB based on their own style.

- No inconsistencies observed.

- A unique check will be required to verify and cross check the content to the expected values.

**lease_commence_date**
- Metadata Reference:
> The year the lease commenced

- No inconsistencies observed.

**remaining_lease**
- Metadata Reference:
> Singapore HDB has a lease of 99 years. This column data has quite a few NULL values, and it is calculated based on different years. You may need to adjust this column data accordingly when building the model.

- Data standardisation will need to be applied as there are various entries which utilises short forms instead of the full form (i.e., 62 y 05 months).

- Data entry which contains no months should have '0 months' instead of leaving the month field empty entirely.

- A data format standardisation approach may be required to upkeep the quality of the column where we change the current data entry format of no capitalisation to a data entry format of first-letter capitalisation (i.e., 'years' to 'Years').

**resale_price**
- Metadata Reference:
> The resale price of the flat.

- No inconsistencies observed.

With the observations indicated above, we will proceed to conduct the Missing Values check in the next section.

---
# 3. Missing Values Identification

In this section, we will be indentifying any missing values in the HDB dataset provided. Should there be any missing values identified, we will be required to subsequently identify a suitable approach in handling the missing values depending on the scale of the missing values and what we hope to achieve through the imputation. The identification process will be conducted in the code cell below.

In [ ]:
"""DAVI CA1 — Column-wise Missingness Overview.

Computes and visualises the proportion of missing data per column in the
HDB Resale Flat Prices dataset.
Each row is coloured dark red if the column has missing data, or dark green
if the column is fully complete.
"""

# =========================
# Column-wise Missingness
# =========================
miss_df = (
    pd.DataFrame({
        "column": df.columns,
        "Missing Data Count": df.isna().sum().values,
        "Percentage Missing (%)": (df.isna().mean().values * 100).round(2),
    })
    .sort_values(["Missing Data Count", "Percentage Missing (%)"],
                 ascending=[False, False])
    .reset_index(drop=True)
)


# =========================
# Styling Helper
# =========================
def row_style(row):
    """Return full-row background style based on missingness.

    Dark red if any missing values, dark green if complete.

    Args:
        row: A pandas Series representing one row.

    Returns:
        List of inline CSS style strings (one per column).
    """
    if row["Missing Data Count"] > 0:
        return ["background-color:#8B0000; color:white; font-weight:bold;"] * len(row)
    return ["background-color:#006400; color:white; font-weight:bold;"] * len(row)


# =========================
# Styled Table Construction
# =========================
styled_missingness = (
    miss_df.style
    .format({"Percentage Missing (%)": "{:.2f}%"})
    .apply(row_style, axis=1)
    .set_properties(**{
        "border": "1px solid #444",
        "font-size": "12px",
        "color": "#e8e8e8",
    })
    .set_table_styles([
        {
            "selector": "th",
            "props": [
                ("background-color", "#1f2c3b"),
                ("color", "#f1f1f1"),
                ("border", "1px solid #555"),
                ("font-weight", "bold"),
            ],
        },
        {
            "selector": "td",
            "props": [
                ("padding", "6px 8px"),
            ],
        },
    ])
)


# =========================
# Display DataFrame
# =========================
display(styled_missingness)

With reference to the output of the code cell above, we are able to confirm and identify that there are no missing values within the provided dataset. Thereforem we are able to procced to the next section where we will indentify and address any duplicated data within our dataset.

---
# 4. Identifying Duplicated Data

In this section, we will be identifying and addressing any duplicated data within our dataset. Duplicated data can cause multiple issues such as influencing the numerical summaries of our data and visualisation. Thereforem, should there be any duplicated data identified within our dataset, we will have to drop them.

In [ ]:
"""DAVI CA1 — Duplicate Row Analysis.

Checks for duplicate rows across all columns in the HDB Resale Flat Prices
dataset. A red flag table is displayed if duplicates exist; otherwise, a
green confirmation table is shown.
"""

# =========================
# Duplicate Count
# =========================
all_cols = df.columns.tolist()
dup_rows = int(df.duplicated(subset=all_cols, keep=False).sum())

summary = pd.DataFrame(
    [{"Metric": "Duplicated Rows", "Count": dup_rows}]
)


# =========================
# Styling Helpers
# =========================
def style_dark_red_flag(df_in):
    """Return a red-styled table for duplicate warnings.

    Args:
        df_in: DataFrame containing metric results.

    Returns:
        Styled DataFrame with dark red background and white text.
    """
    return (
        df_in.style
        .set_properties(**{
            "background-color": "#8B0000",
            "color": "white",
            "font-weight": "bold",
            "border": "1px solid #444",
            "font-size": "12px"
        })
        .set_table_styles([
            {
                "selector": "th",
                "props": [
                    ("background-color", "#1f2c3b"),
                    ("color", "#f1f1f1"),
                    ("border", "1px solid #555"),
                    ("font-weight", "bold")
                ]
            }
        ])
    )


def style_full_green(df_in):
    """Return a green-styled table when no duplicates are found.

    Args:
        df_in: DataFrame containing metric results.

    Returns:
        Styled DataFrame with dark green background and white text.
    """
    return (
        df_in.style
        .set_properties(**{
            "background-color": "#006400",
            "color": "white",
            "font-weight": "bold",
            "border": "1px solid #444",
            "font-size": "12px"
        })
        .set_table_styles([
            {
                "selector": "th",
                "props": [
                    ("background-color", "#1f2c3b"),
                    ("color", "#f1f1f1"),
                    ("border", "1px solid #555"),
                    ("font-weight", "bold")
                ]
            }
        ])
    )


# =========================
# Display Result
# =========================
if dup_rows > 0:
    display(style_dark_red_flag(summary))
else:
    display(style_full_green(summary))

With reference to the output of the code celL above, we are able to identify that there are a total of '606' duplicated rows within out dataset. Therefore, we will need to address the duplicated data by dropping these duplicated rows.

---
## 4.1 Addressing Duplicated Data

In this sub-section, we will be addressing the previously identified duplicated data and we will be dropping them. Subsequently, we will be re-evaluating to confirm that the data have been dropped successfully and we are able to proceed to the next step of the Data Cleaning.

In [ ]:
"""DAVI CA1 — Drop Exact Duplicates Across All Columns.

Removes all exact duplicate rows from the HDB Resale Flat Prices dataset,
then rechecks for remaining duplicates.
Displays a green flag if dataset is clean, or a red warning if duplicates remain.
"""

# =========================
# Drop Exact Duplicates
# =========================
before = len(df)
df = df.drop_duplicates(keep="first").reset_index(drop=True)
after = len(df)

# =========================
# Recheck for Remaining Duplicates
# =========================
dup_rows = int(df.duplicated(keep=False).sum())
summary = pd.DataFrame([
    {"Metric": "Duplicated Rows", "Count": dup_rows},
    {"Metric": "Rows Before", "Count": before},
    {"Metric": "Rows After", "Count": after},
    {"Metric": "Rows Removed", "Count": before - after},
])


# =========================
# Display Summary
# =========================
if dup_rows > 0:
    display(style_dark_red_flag(summary))
else:
    display(style_full_green(summary))

With reference to the code cell above, we are able to verify that the duplicated data have been successfully addressed and we are able to proceed to conduct inspection on the data type for the various features within the dataset.

---
# 5. Data Types Identification

In this section, we will be identifying the various Data Types that each of the Features of the dataset are in. This process will help us in identifying and addressing any data abbornalies later on. This is because with reference to the metadata, we will have a understanding of the intended data type of the feature.

In [ ]:
"""DAVI CA1 — Compact Dark-Styled Column Summary.

Generates a concise summary of all columns in the HDB Resale Flat Prices
dataset, displaying each column’s data type, number of unique values, and
an example entry.
Includes dark-theme styling with colour cues by dtype:
• Numeric → dark green
• Datetime → dark blue
• Boolean → purple
• Object / Category → dark grey
"""

# =========================
# Column Summary Table
# =========================
cols = list(df.columns)

summary = pd.DataFrame({
    "column": cols,
    "dtype": [df[c].dtype for c in cols],
    "n_unique": [df[c].nunique(dropna=True) for c in cols],
    "example_value": [
        df[c].dropna().iloc[0] if df[c].notna().any() else "" for c in cols
    ],
})


# =========================
# Styling Function
# =========================
def style_info_table(df_in):
    """Return a dark-styled summary table with dtype-based colouring.

    Args:
        df_in: DataFrame containing column metadata.

    Returns:
        Styled DataFrame with custom colour cues by dtype category.
    """
    sty = (
        df_in.style
        .set_properties(**{"border": "1px solid #333", "font-size": "12px"})
        .set_table_styles([
            {
                "selector": "th",
                "props": [
                    ("background-color", "#1f2c3b"),
                    ("color", "white"),
                    ("border", "1px solid #333"),
                    ("font-weight", "bold"),
                ],
            },
            {
                "selector": "td",
                "props": [("border", "1px solid #333")],
            },
        ])
        .format({"n_unique": "{:,}"})
    )


    # =========================
    # Dtype Colour Coding
    # =========================
    def colour_dtype(val):
        """Colour the dtype column according to data type."""
        t = str(val)
        if any(x in t for x in ["int", "float", "Int", "Float"]):
            return "background-color:#006400; color:white; font-weight:bold;"
        if "datetime64" in t or "datetime" in t:
            return "background-color:#0b3066; color:white; font-weight:bold;"
        if "bool" in t:
            return "background-color:#4b3f72; color:white; font-weight:bold;"
        return "background-color:#2b2b2b; color:#eaeaea;"

    sty = sty.map(colour_dtype, subset=pd.IndexSlice[:, ["dtype"]])


    # =========================
    # Cell-Specific Styling
    # =========================
    sty = sty.set_properties(
        subset=["example_value"],
        **{"background-color": "#262626", "color": "#eaeaea"},
    )
    sty = sty.set_properties(
        subset=["column"],
        **{"background-color": "#1e1e1e", "color": "#eaeaea", "font-weight": "bold"},
    )

    return sty


# =========================
# Display Styled Summary
# =========================
display(style_info_table(summary))

With reference to the output of the code cell above, we are able to verify the data types of the various features within the dataset. This subsequently tells us that there are data inconsistencies in data columns such as 'flat_type' which will need to be addressed in the next section.

---
# 6. Data Abnormalies and Inconsistencies

In this section, we will be identifying and addressing the various data abnormalieis and inconsistencies within the HDB dataset provided. These inspections will be conducted at the feature level and subsequently, should there be any data issues identified, we will provide an action plan so as to maintain the data integrity and clean the data with respect to the metadata provided and / or the unique data within that feature.

---
## 6.1 Abnormalies Identification in 'Month' Data Column

In this sub-section, we will be indentifying if there is any Data Abnormalies within the 'month' data column. Should there be any abnormalies, we will need to subsequently address the issues identified in another code cell. The strategy that we will utilise to identify abnormal data is summing each unique value of the data and sorting them in ascending order. This will allow us to look at 'rarer' values which will most likely be abnormalies as they are often one-off incidents.

In [ ]:
"""DAVI CA1 — Value Distribution (Counts) with Red Flags.

Builds a frequency table for a chosen column (here: 'month') and styles it
for dark mode. Uses a custom deep red gradient where rarer categories appear
darker red, and higher counts remain readable in muted tones — never white.
"""

# =========================
# Styling Helper
# =========================
def style_count_red_flag(df_in: pd.DataFrame, count_col: str = "count"):
    """Return a dark-themed table with a custom dark red gradient on counts.

    The lower the count, the darker the red, to flag rare categories.
    The higher the count, the lighter (but still dark-friendly) shade.

    Args:
        df_in: DataFrame containing the value counts.
        count_col: Name of the numeric column to colour by (default: "count").

    Returns:
        A styled DataFrame suitable for notebook display.
    """
    # Validate column and coerce numeric
    if count_col not in df_in.columns:
        raise KeyError(f"'{count_col}' not found in DataFrame columns: {list(df_in.columns)}")
    df_num = df_in.copy()
    df_num[count_col] = pd.to_numeric(df_num[count_col], errors="coerce").fillna(0)

    # Custom dark red colormap (deep maroon → medium red)
    dark_red_cmap = mpl.colors.LinearSegmentedColormap.from_list(
        "dark_red_cmap", ["#330000", "#660000", "#8B0000", "#AA1C1C", "#CC3333"]
    )

    sty = (
        df_num.style
        .background_gradient(cmap=dark_red_cmap, subset=[count_col], axis=None)
        .set_properties(**{
            "border": "1px solid #333",
            "font-size": "12px",
            "color": "#f0f0f0",
            "font-weight": "bold",
        })
        .set_table_styles([
            {
                "selector": "th",
                "props": [
                    ("background-color", "#1f2c3b"),
                    ("color", "white"),
                    ("border", "1px solid #333"),
                    ("font-weight", "bold"),
                ],
            },
            {
                "selector": "td",
                "props": [("border", "1px solid #333")],
            },
        ])
        .format({count_col: "{:,}"})
    )
    return sty


# =========================
# Inspect Unique Month Values
# =========================
if "month" in df.columns:
    month_df = (
        df["month"]
        .value_counts(dropna=False)
        .rename_axis("month")
        .reset_index(name="count")
        .sort_values("count", ascending=True, ignore_index=True)
    )

    display(style_count_red_flag(month_df, count_col="count"))
else:
    print("Column 'month' not found in DataFrame.")

With reference to the output of the code cell above, we are able to make the following conclusions and observatios on the data abnormalies.

- There are no Data Abnormalies observed

With the observation indicated above, we will proceed to identify the data abnormalies.

---
## 6.2 Abnormalies Identification in 'Town' Data Column

In this sub-section, we will be indentifying if there is any Data Abnormalies within the 'town' data column. Should there be any abnormalies, we will need to subsequently address the issues identified in another code cell. The strategy that we will utilise to identify abnormal data is summing each unique value of the data and sorting them in ascending order. This will allow us to look at 'rarer' values which will most likely be abnormalies as they are often one-off incidents.

In [ ]:
"""DAVI CA1 — Value Distribution (Counts) with Red Flags.

Builds a frequency table for a chosen column (here: 'town') and styles it
for dark mode. Low-frequency categories are highlighted with a reversed
red gradient (darker red = rarer), enabling quick anomaly/rarity scans.
"""

# =========================
# Inspect Unique Town Values
# =========================
if "town" in df.columns:
    town_df = (
        df["town"]
        .value_counts(dropna=False)
        .rename_axis("town")
        .reset_index(name="count")
        .sort_values("count", ascending=True, ignore_index=True)
    )
    display(style_count_red_flag(town_df, count_col="count"))
else:
    print("Column 'town' not found in DataFrame.")

With reference to the output of the code cell above, we are able to make the following conclusions and observatios on the data abnormalies.

- There are 1 Data Abnormalies observed

**Action Plan:**

1. Standardise all Data Entry Format to First-Letter Capitalisation (i.e., 'ANG MO KIO' to 'Ang Mo Kio')


**Justification**

1. By standardising all data format, data is able to be viewed easier on visualisation axis.


With the action plan and justification indicated above, we will proceed to address the data abnormalies.

---
### 6.2.1 Abnormalies Addressment in 'Town' Data Column

In this sub-sub section, we will be addressing the data abnormalies we previously identified. We will carry out the action plan we indicated and subsequently verify if the changes have been applied to the Data Column.

In [ ]:
"""DAVI CA1 — Fix Town Column and Display Distribution.

Quickly corrects specific malformed entries in the 'town' column, then
display a frequency table with a green gradient to
highlight rarer categories.
"""

# =========================
# Normalise and Map Variants
# =========================
if "town" not in df.columns:
    raise KeyError("Column 'town' not found in DataFrame.")

# Normalise whitespace/case, apply mapping, then convert to Proper Case
df["town"] = (
    df["town"]
    .astype(str)
    .str.strip()
    .str.replace(r"\s+", " ", regex=True)
    .str.upper()
    .str.title()
)

# =========================
# Verify via Display
# =========================
town_df_cleaned = (
    df["town"]
    .value_counts(dropna=False)
    .rename_axis("town")
    .reset_index(name="count")
    .sort_values("count", ascending=True, ignore_index=True)
)

# =========================
# Display DataFrame
# =========================
display(style_full_green(town_df_cleaned))

With reference to the output of the code cell above, we are able to verify that the action plan previously mentioned have been successfully carried out and we are able to proceed to indentify and address other data abnormalies in the other features.

---
## 6.3 Abnormalies Identification in 'Blk_no' Data Column

In this sub-section, we will be indentifying if there is any Data Abnormalies within the 'blk_no' data column. Should there be any abnormalies, we will need to subsequently address the issues identified in another code cell. The strategy that we will utilise to identify abnormal data is summing each unique value of the data and sorting them in ascending order. This will allow us to look at 'rarer' values which will most likely be abnormalies as they are often one-off incidents.

In [ ]:
"""DAVI CA1 — Inspect Unique 'blk_no' Values (Red Flags for Low Counts).

Generates a frequency table of all unique flat types and applies the
existing dark red flag style to highlight rarer categories.
"""

# =========================
# Inspect Unique Flat Types
# =========================
if "blk_no" in df.columns:
    flat_type_df = (
        df["blk_no"]
        .value_counts(dropna=False)
        .rename_axis("blk_no")
        .reset_index(name="count")
        .sort_values("count", ascending=True, ignore_index=True)
    )

    display(style_count_red_flag(flat_type_df))
else:
    print("Column 'flat_type' not found in DataFrame.")

With reference to the output of the code cell above, we are able to make the following conclusions and observatios on the data abnormalies.

- There are no Data Abnormalies observed

With the observation indicated above, we will proceed to identify the data abnormalies.

---
## 6.4 Abnormalies Identification in 'Road_name' Data Column

In this sub-section, we will be indentifying if there is any Data Abnormalies within the 'road_name' data column. Should there be any abnormalies, we will need to subsequently address the issues identified in another code cell. The strategy that we will utilise to identify abnormal data is summing each unique value of the data and sorting them in ascending order. This will allow us to look at 'rarer' values which will most likely be abnormalies as they are often one-off incidents.

In [ ]:
"""DAVI CA1 — Inspect Unique 'road_name' Values (Red Flags for Low Counts).

Generates a frequency table of all unique road_name identifiers in the dataset
and applies the existing dark red flag style to highlight rarer or
underrepresented entries.
"""

# =========================
# Inspect Unique Road_name Values
# =========================
if "road_name" in df.columns:
    road_name_df = (
        df["road_name"]
        .value_counts(dropna=False)
        .rename_axis("road_name")
        .reset_index(name="count")
        .sort_values("count", ascending=True, ignore_index=True)
    )

    display(style_count_red_flag(road_name_df))
else:
    print("Column 'road_name' not found in DataFrame.")

With reference to the output of the code cell above, we are able to make the following conclusions and observatios on the data abnormalies.

- There are 1 Data Abnormalies observed

**Action Plan:**

1. Standardise all Data Entry Format to First-Letter Capitalisation (i.e., 'ANG MO KIO' to 'Ang Mo Kio')


**Justification**

1. By standardising all data format, data is able to be viewed easier on visualisation axis.

With the action plan and justification indicated above, we will proceed to address the data abnormalies.

---
### 6.4.1 Abnormalies Addressment in 'road_name' Data Column

In this sub-sub section, we will be addressing the data abnormalies we previously identified. We will carry out the action plan we indicated and subsequently verify if the changes have been applied to the Data Column.

In [ ]:
"""DAVI CA1 — Fix road_name Column and Display Distribution.

Quickly corrects specific malformed entries in the 'road_name' column, then
display a frequency table with a green gradient to
highlight rarer categories.
"""

# =========================
# Normalise and Map Variants
# =========================
if "road_name" not in df.columns:
    raise KeyError("Column 'road_name' not found in DataFrame.")

# Normalise whitespace/case, apply mapping, then convert to Proper Case
df["road_name"] = (
    df["road_name"]
    .astype(str)
    .str.strip()
    .str.replace(r"\s+", " ", regex=True)
    .str.upper()
    .str.title()
)

# =========================
# Verify via Display
# =========================
road_name_df_cleaned = (
    df["road_name"]
    .value_counts(dropna=False)
    .rename_axis("road_name")
    .reset_index(name="count")
    .sort_values("count", ascending=True, ignore_index=True)
)

# =========================
# Display DataFrame
# =========================
display(style_full_green(road_name_df_cleaned))

With reference to the output of the code cell above, we are able to verify that the action plan previously mentioned have been successfully carried out and we are able to proceed to indentify and address other data abnormalies in the other features.

---
## 6.5 Abnormalies Identification in 'Building' Data Column

In this sub-section, we will be indentifying if there is any Data Abnormalies within the 'building' data column. Should there be any abnormalies, we will need to subsequently address the issues identified in another code cell. The strategy that we will utilise to identify abnormal data is summing each unique value of the data and sorting them in ascending order. This will allow us to look at 'rarer' values which will most likely be abnormalies as they are often one-off incidents.

In [ ]:
"""DAVI CA1 — Inspect Unique 'building' Values (Red Flags for Low Counts).

Generates a frequency table of all unique street names in the dataset and applies
the existing dark red flag style to highlight streets with very few occurrences.
This helps surface rare or potentially erroneous entries.
"""

# =========================
# Inspect Unique Street Names
# =========================
if "building" in df.columns:
    building_df = (
        df["building"]
        .value_counts(dropna=False)
        .rename_axis("building")
        .reset_index(name="count")
        .sort_values("count", ascending=True, ignore_index=True)
    )

    display(style_count_red_flag(building_df))
else:
    print("Column 'building' not found in DataFrame.")


With reference to the output of the code cell above, we are able to make the following conclusions and observatios on the data abnormalies.

- There are 1 Data Abnormalies observed

**Action Plan:**

1. Standardise all Data Entry Format to First-Letter Capitalisation (i.e., 'ANG MO KIO' to 'Ang Mo Kio')


**Justification**

1. By standardising all data format, data is able to be viewed easier on visualisation axis.

With the action plan and justification indicated above, we will proceed to address the data abnormalies.

---
### 6.5.1 Abnormalies Addressment in 'Building' Data Column

In this sub-sub section, we will be addressing the data abnormalies we previously identified. We will carry out the action plan we indicated and subsequently verify if the changes have been applied to the Data Column.

In [ ]:
"""DAVI CA1 — Fix building Column and Display Distribution.

Quickly corrects specific malformed entries in the 'building' column, then
display a frequency table with a green gradient to
highlight rarer categories.
"""

# =========================
# Normalise and Map Variants
# =========================
if "building" not in df.columns:
    raise KeyError("Column 'building' not found in DataFrame.")

# Normalise whitespace/case, apply mapping, then convert to Proper Case
df["building"] = (
    df["building"]
    .astype(str)
    .str.strip()
    .str.replace(r"\s+", " ", regex=True)
    .str.upper()
    .str.title()
)

# =========================
# Verify via Display
# =========================
building_df_cleaned = (
    df["building"]
    .value_counts(dropna=False)
    .rename_axis("building")
    .reset_index(name="count")
    .sort_values("count", ascending=True, ignore_index=True)
)

# =========================
# Display DataFrame
# =========================
display(style_full_green(building_df_cleaned))

With reference to the output of the code cell above, we are able to verify that the action plan previously mentioned have been successfully carried out and we are able to proceed to indentify and address other data abnormalies in the other features.

---
## 6.6 Abnormalies Identification in 'postal' Data Column

In this sub-section, we will be indentifying if there is any Data Abnormalies within the 'postal' data column. Should there be any abnormalies, we will need to subsequently address the issues identified in another code cell. The strategy that we will utilise to identify abnormal data is summing each unique value of the data and sorting them in ascending order. This will allow us to look at 'rarer' values which will most likely be abnormalies as they are often one-off incidents.

In [ ]:
"""DAVI CA1 — Inspect Unique 'postal' Values (Red Flags for Low Counts).

Generates a frequency table of all unique postal in the dataset
and applies the existing dark red flag style to highlight rarer or
less common postal for quality assurance.
"""

# =========================
# Inspect Unique Postal Code
# =========================
if "postal" in df.columns:
    postal_df = (
        df["postal"]
        .value_counts(dropna=False)
        .rename_axis("postal")
        .reset_index(name="count")
        .sort_values("count", ascending=True, ignore_index=True)
    )

    display(style_count_red_flag(postal_df))
else:
    print("Column 'postal' not found in DataFrame.")

With reference to the output of the code cell above, we are able to make the following conclusions and observatios on the data abnormalies.

- There are no Data Abnormalies observed

With the observation indicated above, we will proceed to identify other data abnormalies.

---
## 6.7 Abnormalies Identification in 'Storey_range' Data Column

In this sub-section, we will be indentifying if there is any Data Abnormalies within the 'storey_range' data column. Should there be any abnormalies, we will need to subsequently address the issues identified in another code cell. The strategy that we will utilise to identify abnormal data is summing each unique value of the data and sorting them in ascending order. This will allow us to look at 'rarer' values which will most likely be abnormalies as they are often one-off incidents.

In [ ]:
"""DAVI CA1 — Inspect Unique 'storey_range' Values (Red Flags for Low Counts).

Generates a frequency table of all unique storey_range in the dataset
and applies the existing dark red flag style to highlight rarer models.
This helps identify uncommon or legacy HDB flat storey range.
"""

# =========================
# Inspect Unique Flat Models
# =========================
if "storey_range" in df.columns:
    storey_range_df = (
        df["storey_range"]
        .value_counts(dropna=False)
        .rename_axis("storey_range")
        .reset_index(name="count")
        .sort_values("count", ascending=True, ignore_index=True)
    )

    display(style_count_red_flag(storey_range_df))
else:
    print("Column 'storey_range' not found in DataFrame.")


With reference to the output of the code cell above, we are able to make the following conclusions and observatios on the data abnormalies.

- There is 1 Data Abnormalies observed

**Action Plan:**

1. Standardise all Data Entry Format to No Capitalisation (i.e., '04 TO 06' to '04 to 06')

**Justification**

1. By standardising all data format, data is able to be viewed easier on visualisation axis.

With the action plan and justification indicated above, we will proceed to address the data abnormalies.

---
### 6.7.1 Abnormalies Addressment in 'Storey_range' Data Column

In this sub-sub section, we will be addressing the data abnormalies we previously identified. We will carry out the action plan we indicated and subsequently verify if the changes have been applied to the Data Column.

In [ ]:
"""DAVI CA1 — Clean & Standardise 'storey_range' Values (In-Place).

Normalises storey range data to a consistent 'N to N' format.
Handles common junk entries, replaces inconsistent separators,
and ensures all values follow a clean, human-readable structure.
Displays final cleaned unique values in solid dark green.
"""

# =========================
# Standardise Format
# =========================
def normalise_storey_text(s: str) -> str:
    """Convert 'NN TO NN' → 'N to N' (lowercase 'to', no leading zeros)."""
    if pd.isna(s):
        return None
    s = s.strip().upper()
    if not re.fullmatch(r"\d{1,2}\s*TO\s*\d{1,2}", s):
        return s  # leave unexpected patterns as-is
    lo, hi = re.findall(r"\d{1,2}", s)
    lo, hi = int(lo), int(hi)
    return f"{lo} to {hi}"


df["storey_range"] = df["storey_range"].map(normalise_storey_text)
df["storey_range"] = df["storey_range"].replace({"": None, "NAN": None})


# =========================
# Inspect Cleaned Unique Values
# =========================
storey_cleaned = (
    df["storey_range"]
    .value_counts(dropna=False)
    .rename_axis("storey_range")
    .reset_index(name="count")
    .sort_values("count", ascending=True, ignore_index=True)
)


# =========================
# Display DataFrame
# =========================
display(style_full_green(storey_cleaned))

With reference to the output of the code cell above, we are able to verify that the action plan previously mentioned have been successfully carried out and we are able to proceed to indentify and address other data abnormalies in the other features.

---
## 6.8 Abnormalies Identification in 'Flat_type' Data Column

In this sub-section, we will be indentifying if there is any Data Abnormalies within the 'Flat_type' data column. Should there be any abnormalies, we will need to subsequently address the issues identified in another code cell. The strategy that we will utilise to identify abnormal data is summing each unique value of the data and sorting them in ascending order. This will allow us to look at 'rarer' values which will most likely be abnormalies as they are often one-off incidents.

In [ ]:
"""DAVI CA1 — Inspect Unique 'flat_type' Values (Red Flags for Low Counts).

Generates a frequency table of all unique flat types and applies the
existing dark red flag style to highlight rarer categories.
"""

# =========================
# Inspect Unique Flat Types
# =========================
if "flat_type" in df.columns:
    flat_type_df = (
        df["flat_type"]
        .value_counts(dropna=False)
        .rename_axis("flat_type")
        .reset_index(name="count")
        .sort_values("count", ascending=True, ignore_index=True)
    )

    display(style_count_red_flag(flat_type_df))
else:
    print("Column 'flat_type' not found in DataFrame.")

With reference to the output of the code cell above, we are able to make the following conclusions and observatios on the data abnormalies.

- There are 1 Data Abnormalies observed

**Action Plan:**

1. Standardise all Data Entry Format to First-Letter Capitalisation (i.e., 'ANG MO KIO' to 'Ang Mo Kio')


**Justification**

1. By standardising all data format, data is able to be viewed easier on visualisation axis.

With the action plan and justification indicated above, we will proceed to address the data abnormalies.

---
### 6.8.1 Abnormalies Addressment in 'Flat_type' Data Column

In this sub-sub section, we will be addressing the data abnormalies we previously identified. We will carry out the action plan we indicated and subsequently verify if the changes have been applied to the Data Column.

In [ ]:
"""DAVI CA1 — Fix flat_type Column and Display Distribution.

Quickly corrects specific malformed entries in the 'flat_type' column, then
display a frequency table with a green gradient to
highlight rarer categories.
"""

# =========================
# Normalise and Map Variants
# =========================
if "flat_type" not in df.columns:
    raise KeyError("Column 'flat_type' not found in DataFrame.")

# Normalise whitespace/case, apply mapping, then convert to Proper Case
df["flat_type"] = (
    df["flat_type"]
    .astype(str)
    .str.strip()
    .str.replace(r"\s+", " ", regex=True)
    .str.upper()
    .str.title()
)

# =========================
# Verify via Display
# =========================
flat_type_df_cleaned = (
    df["flat_type"]
    .value_counts(dropna=False)
    .rename_axis("flat_type")
    .reset_index(name="count")
    .sort_values("count", ascending=True, ignore_index=True)
)

# =========================
# Display DataFrame
# =========================
display(style_full_green(flat_type_df_cleaned))

With reference to the output of the code cell above, we are able to verify that the action plan previously mentioned have been successfully carried out and we are able to proceed to indentify and address other data abnormalies in the other features.

---
## 6.9 Abnormalies Identification in 'Flat_model' Data Column

In this sub-section, we will be indentifying if there is any Data Abnormalies within the 'flat_model' data column. Should there be any abnormalies, we will need to subsequently address the issues identified in another code cell. The strategy that we will utilise to identify abnormal data is summing each unique value of the data and sorting them in ascending order. This will allow us to look at 'rarer' values which will most likely be abnormalies as they are often one-off incidents.

In [ ]:
"""DAVI CA1 — Fix flat_model Column and Display Distribution.

Quickly corrects specific malformed entries in the 'flat_model' column, then
display a frequency table with a green gradient to
highlight rarer categories.
"""

# =========================
# Normalise and Map Variants
# =========================
if "flat_model" not in df.columns:
    raise KeyError("Column 'flat_model' not found in DataFrame.")

# Normalise whitespace/case, apply mapping, then convert to Proper Case
df["flat_model"] = (
    df["flat_model"]
    .astype(str)
    .str.strip()
    .str.replace(r"\s+", " ", regex=True)
    .str.upper()
    .str.title()
)

# =========================
# Verify via Display
# =========================
flat_model_df_cleaned = (
    df["flat_model"]
    .value_counts(dropna=False)
    .rename_axis("flat_model")
    .reset_index(name="count")
    .sort_values("count", ascending=True, ignore_index=True)
)

# =========================
# Display DataFrame
# =========================
display(style_full_green(flat_model_df_cleaned))

With reference to the output of the code cell above, we are able to make the following conclusions and observatios on the data abnormalies.

- There are no Data Abnormalies observed

With the observation indicated above, we will proceed to identify other data abnormalies.

---
## 6.10 Abnormalies Identification in 'Lease_commence_date' Data Column

In this sub-section, we will be indentifying if there is any Data Abnormalies within the 'lease_commence_date' data column. Should there be any abnormalies, we will need to subsequently address the issues identified in another code cell. The strategy that we will utilise to identify abnormal data is summing each unique value of the data and sorting them in ascending order. This will allow us to look at 'rarer' values which will most likely be abnormalies as they are often one-off incidents.

In [ ]:
"""DAVI CA1 — Inspect Unique 'lease_commence_date' Values (Red Flags for Low Counts).

Generates a frequency table of all unique flat types and applies the
existing dark red flag style to highlight rarer categories.
"""

# =========================
# Inspect Unique Flat Types
# =========================
if "lease_commence_date" in df.columns:
    lease_commence_date_df = (
        df["lease_commence_date"]
        .value_counts(dropna=False)
        .rename_axis("lease_commence_date")
        .reset_index(name="count")
        .sort_values("count", ascending=True, ignore_index=True)
    )

    display(style_count_red_flag(lease_commence_date_df))
else:
    print("Column 'lease_commence_date' not found in DataFrame.")

With reference to the output of the code cell above, we are able to make the following conclusions and observatios on the data abnormalies.

- There are no Data Abnormalies observed

With the observation indicated above, we will proceed to identify other data abnormalies.

---
## 6.11 Abnormalies Identification in 'Remaining_lease_years' Data Column

In this sub-section, we will be indentifying if there is any Data Abnormalies within the 'remaining_lease_years' data column. Should there be any abnormalies, we will need to subsequently address the issues identified in another code cell. The strategy that we will utilise to identify abnormal data is summing each unique value of the data and sorting them in ascending order. This will allow us to look at 'rarer' values which will most likely be abnormalies as they are often one-off incidents.

In [ ]:
"""DAVI CA1 — Inspect Unique 'remaining_lease_years' Values (Red Flags for Low Counts).

Generates a frequency table of all unique flat types and applies the
existing dark red flag style to highlight rarer categories.
"""

# =========================
# Inspect Unique Flat Types
# =========================
if "remaining_lease_years" in df.columns:
    remaining_lease_years_df = (
        df["remaining_lease_years"]
        .value_counts(dropna=False)
        .rename_axis("remaining_lease_years")
        .reset_index(name="count")
        .sort_values("count", ascending=True, ignore_index=True)
    )

    display(style_count_red_flag(remaining_lease_years_df))
else:
    print("Column 'remaining_lease_years' not found in DataFrame.")

With reference to the output of the code cell above, we are able to make the following conclusions and observatios on the data abnormalies.

- There are no Data Abnormalies observed

With the observation indicated above, we will proceed to identify other data abnormalies.

---
## 6.12 Abnormalies Identification in 'Remaining_lease_months' Data Column

In this sub-section, we will be indentifying if there is any Data Abnormalies within the 'remaining_lease_months' data column. Should there be any abnormalies, we will need to subsequently address the issues identified in another code cell. The strategy that we will utilise to identify abnormal data is summing each unique value of the data and sorting them in ascending order. This will allow us to look at 'rarer' values which will most likely be abnormalies as they are often one-off incidents.

In [ ]:
"""DAVI CA1 — Inspect Unique 'remaining_lease_months' Values (Red Flags for Low Counts).

Generates a frequency table of all unique flat types and applies the
existing dark red flag style to highlight rarer categories.
"""

# =========================
# Inspect Unique Flat Types
# =========================
if "remaining_lease_months" in df.columns:
    remaining_lease_months_df = (
        df["remaining_lease_months"]
        .value_counts(dropna=False)
        .rename_axis("remaining_lease_months")
        .reset_index(name="count")
        .sort_values("count", ascending=True, ignore_index=True)
    )

    display(style_count_red_flag(remaining_lease_months_df))
else:
    print("Column 'remaining_lease_months' not found in DataFrame.")

With reference to the output of the code cell above, we are able to make the following conclusions and observatios on the data abnormalies.

- There are no Data Abnormalies observed

With the observation indicated above, we will proceed to identify other data abnormalies.

---
## 6.13 Abnormalies Identification in 'Planning_area_ura' Data Column

In this sub-section, we will be indentifying if there is any Data Abnormalies within the 'planning_area_ura' data column. Should there be any abnormalies, we will need to subsequently address the issues identified in another code cell. The strategy that we will utilise to identify abnormal data is summing each unique value of the data and sorting them in ascending order. This will allow us to look at 'rarer' values which will most likely be abnormalies as they are often one-off incidents.

In [ ]:
"""DAVI CA1 — Inspect Unique 'planning_area_ura' Values (Red Flags for Low Counts).

Generates a frequency table of all unique flat types and applies the
existing dark red flag style to highlight rarer categories.
"""

# =========================
# Inspect Unique Flat Types
# =========================
if "planning_area_ura" in df.columns:
    planning_area_ura_df = (
        df["planning_area_ura"]
        .value_counts(dropna=False)
        .rename_axis("planning_area_ura")
        .reset_index(name="count")
        .sort_values("count", ascending=True, ignore_index=True)
    )

    display(style_count_red_flag(planning_area_ura_df))
else:
    print("Column 'planning_area_ura' not found in DataFrame.")

With reference to the output of the code cell above, we are able to make the following conclusions and observatios on the data abnormalies.

- There are 1 Data Abnormalies observed

**Action Plan:**

1. Standardise all Data Entry Format to First-Letter Capitalisation (i.e., 'ANG MO KIO' to 'Ang Mo Kio')


**Justification**

1. By standardising all data format, data is able to be viewed easier on visualisation axis.

With the action plan and justification indicated above, we will proceed to address the data abnormalies.

---
### 6.13.1 Abnormalies Addressment in 'Planning_area_ura' Data Column

In this sub-sub section, we will be addressing the data abnormalies we previously identified. We will carry out the action plan we indicated and subsequently verify if the changes have been applied to the Data Column.

In [ ]:
"""DAVI CA1 — Fix planning_area_ura Column and Display Distribution.

Quickly corrects specific malformed entries in the 'planning_area_ura' column, then
display a frequency table with a green gradient to
highlight rarer categories.
"""

# =========================
# Normalise and Map Variants
# =========================
if "planning_area_ura" not in df.columns:
    raise KeyError("Column 'planning_area_ura' not found in DataFrame.")

# Normalise whitespace/case, apply mapping, then convert to Proper Case
df["planning_area_ura"] = (
    df["planning_area_ura"]
    .astype(str)
    .str.strip()
    .str.replace(r"\s+", " ", regex=True)
    .str.upper()
    .str.title()
)

# =========================
# Verify via Display
# =========================
planning_area_ura_df_cleaned = (
    df["planning_area_ura"]
    .value_counts(dropna=False)
    .rename_axis("planning_area_ura")
    .reset_index(name="count")
    .sort_values("count", ascending=True, ignore_index=True)
)

# =========================
# Display DataFrame
# =========================
display(style_full_green(planning_area_ura_df_cleaned))

With reference to the output of the code cell above, we are able to verify that the action plan previously mentioned have been successfully carried out and we are able to proceed to indentify and address other data abnormalies in the other features.

---
## 6.14 Abnormalies Identification in 'Region_ura' Data Column

In this sub-section, we will be indentifying if there is any Data Abnormalies within the 'region_ura' data column. Should there be any abnormalies, we will need to subsequently address the issues identified in another code cell. The strategy that we will utilise to identify abnormal data is summing each unique value of the data and sorting them in ascending order. This will allow us to look at 'rarer' values which will most likely be abnormalies as they are often one-off incidents.

In [ ]:
"""DAVI CA1 — Inspect Unique 'region_ura' Values (Red Flags for Low Counts).

Generates a frequency table of all unique flat types and applies the
existing dark red flag style to highlight rarer categories.
"""

# =========================
# Inspect Unique Flat Types
# =========================
if "region_ura" in df.columns:
    region_ura_df = (
        df["region_ura"]
        .value_counts(dropna=False)
        .rename_axis("region_ura")
        .reset_index(name="count")
        .sort_values("count", ascending=True, ignore_index=True)
    )

    display(style_count_red_flag(region_ura_df))
else:
    print("Column 'region_ura' not found in DataFrame.")

With reference to the output of the code cell above, we are able to make the following conclusions and observatios on the data abnormalies.

- There are 1 Data Abnormalies observed

**Action Plan:**

1. Standardise all Data Entry Format to First-Letter Capitalisation (i.e., 'ANG MO KIO' to 'Ang Mo Kio')


**Justification**

1. By standardising all data format, data is able to be viewed easier on visualisation axis.

With the action plan and justification indicated above, we will proceed to address the data abnormalies.

---
### 6.14.1 Abnormalies Addressment in 'Region_ura' Data Column

In this sub-sub section, we will be addressing the data abnormalies we previously identified. We will carry out the action plan we indicated and subsequently verify if the changes have been applied to the Data Column.

In [ ]:
"""DAVI CA1 — Fix region_ura Column and Display Distribution.

Quickly corrects specific malformed entries in the 'region_ura' column, then
display a frequency table with a green gradient to
highlight rarer categories.
"""

# =========================
# Normalise and Map Variants
# =========================
if "region_ura" not in df.columns:
    raise KeyError("Column 'region_ura' not found in DataFrame.")

# Normalise whitespace/case, apply mapping, then convert to Proper Case
df["region_ura"] = (
    df["region_ura"]
    .astype(str)
    .str.strip()
    .str.replace(r"\s+", " ", regex=True)
    .str.upper()
    .str.title()
)

# =========================
# Verify via Display
# =========================
region_ura_df_cleaned = (
    df["region_ura"]
    .value_counts(dropna=False)
    .rename_axis("region_ura")
    .reset_index(name="count")
    .sort_values("count", ascending=True, ignore_index=True)
)

# =========================
# Display DataFrame
# =========================
display(style_full_green(region_ura_df_cleaned))

With reference to the output of the code cell above, we are able to verify that the action plan previously mentioned have been successfully carried out and we are able to proceed to indentify and address other data abnormalies in the other features.

---
## 6.15 Abnormalies Identification in 'Closest_mrt_station' Data Column

In this sub-section, we will be indentifying if there is any Data Abnormalies within the 'closest_mrt_station' data column. Should there be any abnormalies, we will need to subsequently address the issues identified in another code cell. The strategy that we will utilise to identify abnormal data is summing each unique value of the data and sorting them in ascending order. This will allow us to look at 'rarer' values which will most likely be abnormalies as they are often one-off incidents.

In [ ]:
"""DAVI CA1 — Inspect Unique 'closest_mrt_station' Values (Red Flags for Low Counts).

Generates a frequency table of all unique flat types and applies the
existing dark red flag style to highlight rarer categories.
"""

# =========================
# Inspect Unique Flat Types
# =========================
if "closest_mrt_station" in df.columns:
    closest_mrt_station_df = (
        df["closest_mrt_station"]
        .value_counts(dropna=False)
        .rename_axis("closest_mrt_station")
        .reset_index(name="count")
        .sort_values("count", ascending=True, ignore_index=True)
    )

    display(style_count_red_flag(closest_mrt_station_df))
else:
    print("Column 'closest_mrt_station' not found in DataFrame.")

With reference to the output of the code cell above, we are able to make the following conclusions and observatios on the data abnormalies.

- There are no Data Abnormalies observed

With the observation indicated above, we will proceed to identify other data abnormalies.

---
## 6.16 Abnormalies Identification in 'Transport_type' Data Column

In this sub-section, we will be indentifying if there is any Data Abnormalies within the 'transport_type' data column. Should there be any abnormalies, we will need to subsequently address the issues identified in another code cell. The strategy that we will utilise to identify abnormal data is summing each unique value of the data and sorting them in ascending order. This will allow us to look at 'rarer' values which will most likely be abnormalies as they are often one-off incidents.

In [ ]:
"""DAVI CA1 — Inspect Unique 'transport_type' Values (Red Flags for Low Counts).

Generates a frequency table of all unique flat types and applies the
existing dark red flag style to highlight rarer categories.
"""

# =========================
# Inspect Unique Flat Types
# =========================
if "transport_type" in df.columns:
    transport_type_df = (
        df["transport_type"]
        .value_counts(dropna=False)
        .rename_axis("transport_type")
        .reset_index(name="count")
        .sort_values("count", ascending=True, ignore_index=True)
    )

    display(style_count_red_flag(transport_type_df))
else:
    print("Column 'transport_type' not found in DataFrame.")

With reference to the output of the code cell above, we are able to make the following conclusions and observatios on the data abnormalies.

- There are no Data Abnormalies observed

With the observation indicated above, we will proceed to identify other data abnormalies.

---
## 6.17 Abnormalies Identification in 'Line_color' Data Column

In this sub-section, we will be indentifying if there is any Data Abnormalies within the 'line_color' data column. Should there be any abnormalies, we will need to subsequently address the issues identified in another code cell. The strategy that we will utilise to identify abnormal data is summing each unique value of the data and sorting them in ascending order. This will allow us to look at 'rarer' values which will most likely be abnormalies as they are often one-off incidents.

In [ ]:
"""DAVI CA1 — Inspect Unique 'line_color' Values (Red Flags for Low Counts).

Generates a frequency table of all unique flat types and applies the
existing dark red flag style to highlight rarer categories.
"""

# =========================
# Inspect Unique Flat Types
# =========================
if "line_color" in df.columns:
    line_color_df = (
        df["line_color"]
        .value_counts(dropna=False)
        .rename_axis("line_color")
        .reset_index(name="count")
        .sort_values("count", ascending=True, ignore_index=True)
    )

    display(style_count_red_flag(line_color_df))
else:
    print("Column 'line_color' not found in DataFrame.")

With reference to the output of the code cell above, we are able to make the following conclusions and observatios on the data abnormalies.

- There are no Data Abnormalies observed

With the observation indicated above, we will proceed to identify other data abnormalies.

---
## 6.18 Abnormalies Identification in 'Closest_pri_school' Data Column

In this sub-section, we will be indentifying if there is any Data Abnormalies within the 'closest_pri_school' data column. Should there be any abnormalies, we will need to subsequently address the issues identified in another code cell. The strategy that we will utilise to identify abnormal data is summing each unique value of the data and sorting them in ascending order. This will allow us to look at 'rarer' values which will most likely be abnormalies as they are often one-off incidents.

In [ ]:
"""DAVI CA1 — Inspect Unique 'closest_pri_school' Values (Red Flags for Low Counts).

Generates a frequency table of all unique flat types and applies the
existing dark red flag style to highlight rarer categories.
"""

# =========================
# Inspect Unique Flat Types
# =========================
if "closest_pri_school" in df.columns:
    closest_pri_school_df = (
        df["closest_pri_school"]
        .value_counts(dropna=False)
        .rename_axis("closest_pri_school")
        .reset_index(name="count")
        .sort_values("count", ascending=True, ignore_index=True)
    )

    display(style_count_red_flag(closest_pri_school_df))
else:
    print("Column 'closest_pri_school' not found in DataFrame.")

With reference to the output of the code cell above, we are able to make the following conclusions and observatios on the data abnormalies.

- There are 1 Data Abnormalies observed

**Action Plan:**

1. Standardise all Data Entry Format to First-Letter Capitalisation (i.e., 'ANG MO KIO' to 'Ang Mo Kio')


**Justification**

1. By standardising all data format, data is able to be viewed easier on visualisation axis.

With the action plan and justification indicated above, we will proceed to address the data abnormalies.

---
### 6.18.1 Abnormalies Addressment in 'Closest_pri_school' Data Column

In this sub-sub section, we will be addressing the data abnormalies we previously identified. We will carry out the action plan we indicated and subsequently verify if the changes have been applied to the Data Column.

In [ ]:
"""DAVI CA1 — Fix closest_pri_school Column and Display Distribution.

Quickly corrects specific malformed entries in the 'closest_pri_school' column, then
display a frequency table with a green gradient to
highlight rarer categories.
"""

# =========================
# Normalise and Map Variants
# =========================
if "closest_pri_school" not in df.columns:
    raise KeyError("Column 'closest_pri_school' not found in DataFrame.")

# Normalise whitespace/case, apply mapping, then convert to Proper Case
df["closest_pri_school"] = (
    df["closest_pri_school"]
    .astype(str)
    .str.strip()
    .str.replace(r"\s+", " ", regex=True)
    .str.upper()
    .str.title()
)

# =========================
# Verify via Display
# =========================
closest_pri_school_df_cleaned = (
    df["closest_pri_school"]
    .value_counts(dropna=False)
    .rename_axis("closest_pri_school")
    .reset_index(name="count")
    .sort_values("count", ascending=True, ignore_index=True)
)

# =========================
# Display DataFrame
# =========================
display(style_full_green(closest_pri_school_df_cleaned))

With reference to the output of the code cell above, we are able to verify that the action plan previously mentioned have been successfully carried out and we are able to proceed to indentify and address other data abnormalies in the other features.

---
# 7. Outlier Detection

In this section, we will be identifying and addressing Outliers at the univariate level within our dataset. Outliers can cause issues such as skewing numerical summaries and visualisation. Therefore, it is important to identify and address the various outliers within the dataset appropriately.

In [ ]:
"""DAVI CA1 — Boxplots with Outlier Hover Showing 'flat_model'.

Renders one horizontal boxplot per numeric feature. Outlier points show the
associated 'flat_model' in the hover tooltip for quick cross-sectional insight.
"""

# =========================
# Setup Theme
# =========================
pio.templates.default = "plotly_white"


# =========================
# Guard for 'flat_model'
# =========================
if "flat_model" not in df.columns:
    raise KeyError("Column 'flat_model' not found in DataFrame.")


# =========================
# Select Numeric Columns
# =========================
num_cols = df.select_dtypes(include=["number"]).columns.tolist()
if not num_cols:
    raise ValueError("No numeric columns found in DataFrame.")


# =========================
# Render One Chart per Feature (with flat_model in hover)
# =========================
for col in num_cols:
    s = df[col].dropna()
    if s.empty:
        continue

    # Align flat_model to the same index as the numeric series
    flat_model_series = (
        df.loc[s.index, "flat_model"]
        .astype(str)
        .fillna("Unknown")
        .replace({"": "Unknown"})
    )

    d = pd.DataFrame({
        "feature": col,
        "value": s,
        "flat_model": flat_model_series,
    })

    fig = px.box(
        d,
        y="feature",                  # horizontal orientation
        x="value",
        points="outliers",            # show outlier markers only
        color_discrete_sequence=["#3B82F6"],
        custom_data=["flat_model"],   # pass flat_model to hovertemplate
    )

    fig.update_traces(
        boxmean=True,
        marker=dict(size=4, opacity=0.45, line=dict(width=0)),
        # Include flat_model in the hover; appears for outlier points
        hovertemplate="<b>%{y}</b><br>"
                      "Value: %{x:.4g}<br>"
                      "Flat model: %{customdata[0]}<extra></extra>",
    )

    fig.update_layout(
        title=dict(
            text=f"Distribution of <b>{col}</b>",
            x=0.5, xanchor="center",
            font=dict(size=20, family="Inter, Arial"),
        ),
        xaxis_title="Value",
        yaxis_title="",
        font=dict(size=12, family="Inter, Arial"),
        height=320,
        margin=dict(l=90, r=40, t=70, b=40),
        plot_bgcolor="#FAFAFA",
        paper_bgcolor="#FFFFFF",
        hoverlabel=dict(bgcolor="white", font_size=12),
        shapes=[dict(
            type="rect", xref="paper", yref="paper",
            x0=0, y0=0, x1=1, y1=1,
            line=dict(width=0),
            fillcolor="rgba(0,0,0,0)"
        )],
    )

    fig.update_xaxes(showgrid=True, gridcolor="rgba(0,0,0,0.08)", zeroline=False)
    fig.update_yaxes(showgrid=False, zeroline=False)

    fig.show()

With reference to the output of the code cell above, we are able to see that there are outliers for most of the numerical features. However, as these outliers are insignificant, no addressment will be taken into action. Furthermore, we will not be utilising any 'Target' variables from this dataset. Therefore, no addressment of outliers are needed.

---
# 8. Data Export

In this section, we will be exporting the data into CSV so that we are able to utilise this cleaned and processed data for Tableau visualisation purposes. This cleaned dataset will also be subsequently submitted for review as part of the assessment.

In [ ]:
"""DAVI CA1 — Export Cleaned Dataset with Join Key.

This module:
    * Adds a constant-valued join key column to enable Tableau relationships.
    * Exports the fully cleaned and standardised HDB Resale Flat Prices dataset
      to the predefined CLEAN_CSV path.
    * Prints a summary table with export metadata.
"""

# =========================
# Export DataFrame
# =========================
df.to_csv(CLEAN_CSV, index=False, encoding="utf-8-sig")

# =========================
# Verify Export
# =========================
file_size_mb = os.path.getsize(CLEAN_CSV) / (1024 * 1024)

summary = pd.DataFrame(
    [
        {"Metric": "Export Path", "Value": str(CLEAN_CSV)},
        {"Metric": "Rows Exported", "Value": f"{len(df):,}"},
        {"Metric": "Columns", "Value": f"{len(df.columns):,}"},
        {"Metric": "File Size (MB)", "Value": f"{file_size_mb:.2f}"},
    ]
)

try:
    display(style_full_green(summary))
except NameError:
    display(summary)

With reference to the code cell above, we are able to determine that the cleaned dataset have been successfully exported into a new CSV file and we are ready to utilise this CSV file for Data Visualisation in Tableau.